In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Salting-Step1") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 16:42:49 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.190 instead (on interface en0)
26/04/02 16:42:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 16:42:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 16:42:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
hot_key_rows = 10_000_000
normal_rows = 200_000

hot_df = (
    spark.range(hot_key_rows)
    .withColumn("user_id", lit(101))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

normal_df = (
    spark.range(normal_rows)
    .withColumn("user_id", (col("id") % 5000 + 1000).cast("int"))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

transactions_df = hot_df.union(normal_df)

In [3]:
transactions_df.groupBy("user_id").count().orderBy(desc("count")).show(10, False)

+-------+--------+
|user_id|count   |
+-------+--------+
|101    |10000000|
|1003   |40      |
|1021   |40      |
|1030   |40      |
|1044   |40      |
|1088   |40      |
|1092   |40      |
|1108   |40      |
|1160   |40      |
|1195   |40      |
+-------+--------+
only showing top 10 rows


In [4]:
users_df = (
    spark.range(7000)
    .withColumnRenamed("id", "user_id")
    .withColumn("segment", when(col("user_id") == 101, "VIP").otherwise("REGULAR"))
)

In [5]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
joined_df = transactions_df.join(users_df, on="user_id", how="inner")
joined_df.count()

10200000

In [6]:
joined_df.show()

[Stage 16:>                                                         (0 + 1) / 1]

+-------+-------+------+-------+
|user_id| txn_id|amount|segment|
+-------+-------+------+-------+
|    101|      0|   196|    VIP|
|    101|8388608|   240|    VIP|
|    101|      1|   884|    VIP|
|    101|8388609|   320|    VIP|
|    101|      2|   156|    VIP|
|    101|8388610|   693|    VIP|
|    101|      3|   216|    VIP|
|    101|8388611|   211|    VIP|
|    101|      4|   357|    VIP|
|    101|8388612|   945|    VIP|
|    101|      5|   274|    VIP|
|    101|8388613|   965|    VIP|
|    101|      6|   558|    VIP|
|    101|8388614|   651|    VIP|
|    101|      7|    62|    VIP|
|    101|8388615|   590|    VIP|
|    101|      8|   827|    VIP|
|    101|8388616|   738|    VIP|
|    101|      9|   842|    VIP|
|    101|8388617|   525|    VIP|
+-------+-------+------+-------+
only showing top 20 rows


In [7]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 1)

In [8]:
joined_df1 = transactions_df.join(users_df, on="user_id", how="inner")
joined_df1.show()

[Stage 21:>                                                         (0 + 1) / 1]

+-------+-------+------+-------+
|user_id| txn_id|amount|segment|
+-------+-------+------+-------+
|    101|      0|   196|    VIP|
|    101|8388608|   240|    VIP|
|    101|      1|   884|    VIP|
|    101|8388609|   320|    VIP|
|    101|      2|   156|    VIP|
|    101|8388610|   693|    VIP|
|    101|      3|   216|    VIP|
|    101|8388611|   211|    VIP|
|    101|      4|   357|    VIP|
|    101|8388612|   945|    VIP|
|    101|      5|   274|    VIP|
|    101|8388613|   965|    VIP|
|    101|      6|   558|    VIP|
|    101|8388614|   651|    VIP|
|    101|      7|    62|    VIP|
|    101|8388615|   590|    VIP|
|    101|      8|   827|    VIP|
|    101|8388616|   738|    VIP|
|    101|      9|   842|    VIP|
|    101|8388617|   525|    VIP|
+-------+-------+------+-------+
only showing top 20 rows


In [9]:
spark.stop()

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Salting-AQE-OFF") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.adaptive.skewJoin.enabled", "false") \
    .getOrCreate()

print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Skew Join:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

AQE: false
Skew Join: false


26/04/02 16:54:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
hot_key_rows = 10_000_000
normal_rows = 200_000

hot_df = (
    spark.range(hot_key_rows)
    .withColumn("user_id", lit(101))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

normal_df = (
    spark.range(normal_rows)
    .withColumn("user_id", (col("id") % 5000 + 1000).cast("int"))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

transactions_df = hot_df.union(normal_df)

users_df = (
    spark.range(7000)
    .withColumnRenamed("id", "user_id")
    .withColumn("segment", when(col("user_id") == 101, "VIP").otherwise("REGULAR"))
)

In [12]:
joined_df = transactions_df.join(users_df, on="user_id", how="inner")
joined_df.explain(True)
joined_df.count()

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [user_id])
:- Union false, false
:  :- Project [user_id#60, txn_id#61L, amount#62]
:  :  +- Project [id#59L, user_id#60, txn_id#61L, cast((rand(-2536456996570210631) * cast(1000 as double)) as int) AS amount#62]
:  :     +- Project [id#59L, user_id#60, id#59L AS txn_id#61L]
:  :        +- Project [id#59L, 101 AS user_id#60]
:  :           +- Range (0, 10000000, step=1, splits=Some(8))
:  +- Project [user_id#64, txn_id#65L, amount#66]
:     +- Project [id#63L, user_id#64, txn_id#65L, cast((rand(5539173813379975246) * cast(1000 as double)) as int) AS amount#66]
:        +- Project [id#63L, user_id#64, id#63L AS txn_id#65L]
:           +- Project [id#63L, cast(((id#63L % cast(5000 as bigint)) + cast(1000 as bigint)) as int) AS user_id#64]
:              +- Range (0, 200000, step=1, splits=Some(8))
+- Project [user_id#68L, CASE WHEN (user_id#68L = cast(101 as bigint)) THEN VIP ELSE REGULAR END AS segment#69]
   +- Project [id#67L AS user_id#

10200000

In [13]:
from pyspark.sql.functions import spark_partition_id

transactions_df_with_pid = transactions_df.withColumn(
    "partition_id",
    spark_partition_id()
)

transactions_df_with_pid.groupBy("user_id", "partition_id") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, False)

+-------+------------+-------+
|user_id|partition_id|count  |
+-------+------------+-------+
|101    |1           |1250000|
|101    |3           |1250000|
|101    |2           |1250000|
|101    |4           |1250000|
|101    |0           |1250000|
|101    |7           |1250000|
|101    |5           |1250000|
|101    |6           |1250000|
|1000   |8           |5      |
|1009   |8           |5      |
|1002   |8           |5      |
|1003   |8           |5      |
|1010   |8           |5      |
|1031   |8           |5      |
|1015   |8           |5      |
|1004   |8           |5      |
|1029   |8           |5      |
|1032   |8           |5      |
|1043   |8           |5      |
|1011   |8           |5      |
+-------+------------+-------+
only showing top 20 rows
